# SIC8 | 08B v3.0 — Continuous Streaming Silver & CDC Enrichment

**R2 / Continuous Streaming Silver Tier.** This notebook implements the strict Sayed contract parser, isolates ground truth, computes in-stream lag, enriches transactions with CDC user and card dimensions, and reconciles real-time model predictions.

### v3.0 Capabilities:
1. **Continuous Streaming Mode:** Runs continuously (`processingTime='10 seconds'`) for 24/7 low-latency processing, or `availableNow=True` for bounded backfills.
2. **Lag Telemetry:** Computes `ingestion_lag_seconds` (`current_timestamp() - event_time_utc`) and `processing_lag_seconds` (`current_timestamp() - source_modified_utc`).
3. **CDC Dimension Enrichment:** Enriches transactions via broadcast join with `workspace.sic8_silver.historical_users` and `historical_cards` into `workspace.sic8_streaming.streaming_transactions_enriched_v2`.
4. **Model Reconciliation:** Reconciles real-time model predictions (`streaming_model_predictions_v1`) against ground truth (`streaming_synthetic_truth_v2`) to monitor live precision, recall, and drift.


In [0]:
# Configuration — edit RUN_STAGE and STREAMING_MODE as needed.
RUN_STAGE = "SILVER"          # PREFLIGHT | SILVER
STREAMING_MODE = "CONTINUOUS" # CONTINUOUS | AVAILABLE_NOW
TRIGGER_INTERVAL = "10 seconds"

CATALOG = "workspace"
SCHEMA = "sic8_streaming"
RAW_TABLE = f"{CATALOG}.{SCHEMA}.streaming_raw_envelopes_v3"
TX_TABLE = f"{CATALOG}.{SCHEMA}.streaming_transactions_v2"
TRUTH_TABLE = f"{CATALOG}.{SCHEMA}.streaming_synthetic_truth_v2"
QUAR_TABLE = f"{CATALOG}.{SCHEMA}.streaming_quarantine_v2"
REG_TABLE = f"{CATALOG}.{SCHEMA}.streaming_event_registry_v2"

# Enhanced Tier: CDC Enrichment & Predictions Reconciliation
ENRICHED_TX_TABLE = f"{CATALOG}.{SCHEMA}.streaming_transactions_enriched_v2"
PREDICTIONS_TABLE = f"{CATALOG}.{SCHEMA}.streaming_model_predictions_v1"
DRIFT_METRICS_TABLE = f"{CATALOG}.{SCHEMA}.streaming_model_drift_metrics_v1"
DIM_USERS_TABLE = f"{CATALOG}.sic8_silver.historical_users"
DIM_CARDS_TABLE = f"{CATALOG}.sic8_silver.historical_cards"

STATE_VOLUME = f"/Volumes/{CATALOG}/{SCHEMA}/streaming_state"
SILVER_CHECKPOINT = f"{STATE_VOLUME}/silver_transactions_v2/checkpoint"

assert RUN_STAGE in {"PREFLIGHT", "SILVER"}
assert STREAMING_MODE in {"CONTINUOUS", "AVAILABLE_NOW"}
spark.conf.set("spark.sql.session.timeZone", "UTC")
print("RUN_STAGE:", RUN_STAGE)
print("STREAMING_MODE:", STREAMING_MODE, f"(interval={TRIGGER_INTERVAL})")
print("RAW SOURCE:", RAW_TABLE)
print("SILVER TABLES:", TX_TABLE, TRUTH_TABLE, QUAR_TABLE, REG_TABLE)
print("ENRICHED SILVER TABLE:", ENRICHED_TX_TABLE)
print("SILVER CHECKPOINT:", SILVER_CHECKPOINT)


RUN_STAGE: SILVER
RAW: workspace.sic8_streaming.streaming_raw_envelopes_v3
OUTPUT TABLES: workspace.sic8_streaming.streaming_transactions_v2 workspace.sic8_streaming.streaming_synthetic_truth_v2 workspace.sic8_streaming.streaming_quarantine_v2 workspace.sic8_streaming.streaming_event_registry_v2
SILVER CHECKPOINT: /Volumes/workspace/sic8_streaming/streaming_state/silver_transactions_v2/checkpoint


## 1. Source readiness and immutable data contract

The previous 08A Raw table and UC checkpoint volume must exist. Do not create Silver outputs before the diagnostic/fixture tests pass. The raw stream is not modified.

In [0]:
from pyspark.sql import functions as F, types as T

if not spark.catalog.tableExists(RAW_TABLE):
    raise RuntimeError(f"RAW_TABLE_MISSING: {RAW_TABLE}. Complete 08A INGEST first.")
raw_cols = set(spark.table(RAW_TABLE).columns)
required = {"raw_json_line", "source_file", "source_modified_utc", "source_size_bytes",
            "kafka_key", "kafka_offset", "ingested_at_utc"}
missing = sorted(required - raw_cols)
if missing:
    raise RuntimeError(f"RAW_COLUMNS_MISSING: {missing}")
dbutils.fs.ls(STATE_VOLUME)
raw_count = spark.table(RAW_TABLE).count()
if raw_count == 0:
    raise RuntimeError("RAW_EMPTY: 08A INGEST must have inserted raw events")
print("RAW_READY_PASS", raw_count, "rows")

RAW_READY_PASS 275151 rows


## 2. Contract parser — exactly the same parser used for the streaming micro-batches

Uses `from_json(..., MAP<STRING,STRING>)` to handle exact case-sensitive field names with spaces and punctuation. It accepts `value` as an object or an encoded JSON string, and supports `payload` (live) or `data` (legacy). No fabricated Card/User IDs. Only strictly verified Kafka-key fallback for absent User.

In [0]:
def parse_sayed_raw(frame):
    """Pure Spark DataFrame transformation; safe to reuse inside Spark Connect foreachBatch."""
    from pyspark.sql import functions as F, types as T

    # Raw line is Kafka S3 Sink wrapper, or (defensively) an unwrapped event.
    envelope = F.get_json_object(F.col("raw_json_line"), "$.value")
    p = frame.withColumn("_envelope", F.coalesce(envelope, F.when(
        F.get_json_object(F.col("raw_json_line"), "$.event_id").isNotNull(), F.col("raw_json_line"))))
    j = lambda path: F.get_json_object(F.col("_envelope"), path)
    p = (p.withColumn("event_id_parsed", F.trim(j("$.event_id")))
        .withColumn("event_type_parsed", j("$.event_type"))
        .withColumn("schema_version_parsed", j("$.schema_version"))
        .withColumn("source_system", j("$.source_system"))
        .withColumn("run_id", j("$.run_id"))
        .withColumn("scenario", j("$.scenario"))
        .withColumn("event_time_raw", F.coalesce(j("$.event_timestamp"), j("$.timestamp")))
        .withColumn("_business_json", F.coalesce(j("$.payload"), j("$.data"))))
    # Nested transaction shape is only for older payloads; the live schema is flat.
    p = p.withColumn("_business_json", F.coalesce(
        F.get_json_object(F.col("_business_json"), "$.transaction"), F.col("_business_json")))
    p = p.withColumn("_business", F.from_json(F.col("_business_json"),
              T.MapType(T.StringType(), T.StringType(), valueContainsNull=True)))
    get = lambda key: F.element_at(F.col("_business"), F.lit(key))

    # Modern producer: User/Card/Amount/Is Fraud?. Legacy: user_id/card_id/amount/is_fraud.
    p = (p.withColumn("user_raw", F.coalesce(get("User"), get("user_id"), get("userId")))
        .withColumn("card_raw", F.coalesce(get("Card"), get("card_id"), get("card_index"), get("card_idx")))
        .withColumn("amount_raw", F.coalesce(get("Amount"), get("amount"), get("transaction_amount")))
        .withColumn("fraud_raw", F.coalesce(get("Is Fraud?"), get("is_fraud"), get("fraud_label")))
        .withColumn("mcc_raw", F.coalesce(get("MCC"), get("mcc"), get("merchant_category_code")))
        .withColumn("merchant_name", F.coalesce(get("Merchant Name"), get("merchant_name")))
        .withColumn("merchant_city", F.coalesce(get("Merchant City"), get("merchant_city")))
        .withColumn("merchant_state", F.coalesce(get("Merchant State"), get("merchant_state")))
        .withColumn("merchant_lat_raw", F.coalesce(get("Merchant Latitude"), get("merchant_latitude")))
        .withColumn("merchant_lon_raw", F.coalesce(get("Merchant Longitude"), get("merchant_longitude")))
        .withColumn("zip_raw", F.coalesce(get("Zip"), get("zip"), get("zip_code")))
        .withColumn("use_chip", F.coalesce(get("Use Chip"), get("use_chip")))
        .withColumn("transaction_entry_method", F.coalesce(get("Transaction Entry Method"),get("transaction_entry_method")))
        .withColumn("errors", F.coalesce(get("Errors?"), get("errors")))
        .withColumn("currency_raw", F.coalesce(get("currency"), get("Currency")))
        .withColumn("source_transaction_id", F.coalesce(get("transaction_id"), get("tx_id")))
        .withColumn("local_year", get("Year"))
        .withColumn("local_month", get("Month"))
        .withColumn("local_day", get("Day"))
        .withColumn("local_clock", get("Time")))

    p = (p.withColumn("payload_user_numeric", F.expr("try_cast(trim(user_raw) AS BIGINT)"))
        .withColumn("kafka_user_numeric", F.expr("try_cast(trim(kafka_key) AS BIGINT)"))
        .withColumn("user_id", F.when(F.col("user_raw").isNotNull(), F.col("payload_user_numeric"))
                .otherwise(F.col("kafka_user_numeric")))
        .withColumn("user_id_origin", F.when(F.col("user_raw").isNotNull(), F.lit("PAYLOAD"))
                .when(F.col("kafka_user_numeric").isNotNull(), F.lit("KAFKA_KEY_VERIFIED_FALLBACK"))
                .otherwise(F.lit("UNKNOWN")))
        .withColumn("card_index", F.expr("try_cast(trim(card_raw) AS BIGINT)"))
        .withColumn("mcc", F.expr("try_cast(trim(mcc_raw) AS INT)"))
        .withColumn("merchant_latitude", F.expr("try_cast(trim(merchant_lat_raw) AS DOUBLE)"))
        .withColumn("merchant_longitude", F.expr("try_cast(trim(merchant_lon_raw) AS DOUBLE)"))
        .withColumn("zip", F.regexp_replace(F.trim("zip_raw"), r"\.0$", ""))
        .withColumn("event_time_utc", F.expr("try_to_timestamp(event_time_raw)")))

    # Currency has implied USD only in IBM-style $ amount data, NOT converted from EUR.
    p = (p.withColumn("currency", F.when(F.col("currency_raw").isNotNull(), F.upper(F.trim("currency_raw")))
                  .when(F.col("amount_raw").rlike(r"^\s*(?:-?\$|\$-)"), F.lit("USD"))
                  .otherwise(F.lit(None).cast("string")))
        .withColumn("currency_origin", F.when(F.col("currency_raw").isNotNull(), "EXPLICIT")
                  .when(F.col("amount_raw").rlike(r"^\s*(?:-?\$|\$-)"), "IMPLIED_DOLLAR_SIGN")
                  .otherwise("UNKNOWN"))
        .withColumn("amount_stripped", F.regexp_replace(F.trim("amount_raw"), r"[\$,]", ""))
        .withColumn("amount", F.expr("try_cast(amount_stripped AS DECIMAL(18,2))"))
        .withColumn("fraud_norm", F.lower(F.trim("fraud_raw")))
        .withColumn("synthetic_is_fraud", F.when(F.col("fraud_norm").isin("yes", "true", "1"), F.lit(True))
                  .when(F.col("fraud_norm").isin("no", "false", "0"), F.lit(False))
                  .otherwise(F.lit(None).cast("boolean")))
        .withColumn("scoped_card_key", F.concat_ws(":", "user_id", "card_index"))
        .withColumn("kafka_partition", F.expr("try_cast(regexp_extract(source_file, 'partition=([0-9]+)', 1) AS INT)"))
        .withColumn("transaction_id_origin", F.when(F.length(F.trim("source_transaction_id"))>0, "SOURCE")
                  .otherwise("DERIVED_FROM_EVENT_ID"))
        .withColumn("transaction_id", F.when(F.col("transaction_id_origin")=="SOURCE", F.col("source_transaction_id"))
                  .otherwise(F.concat(F.lit("EVENT_"), F.col("event_id_parsed")))))

    # Local timestamp has NO known timezone; don't use it for UTC velocity or GPS fraud checks.
    local_text = F.concat(F.lpad(F.trim("local_year"), 4, "0"), F.lit("-"),
                          F.lpad(F.trim("local_month"), 2, "0"), F.lit("-"),
                          F.lpad(F.trim("local_day"), 2, "0"), F.lit(" "),
                          F.when(F.trim("local_clock").rlike(r"^[0-9]{2}:[0-9]{2}$"),
                                 F.concat(F.trim("local_clock"),F.lit(":00")))
                           .otherwise(F.trim("local_clock")))
    p = p.withColumn("_local_text", local_text).withColumn(
        "source_event_time_local", F.expr("try_cast(_local_text AS TIMESTAMP_NTZ)"))
    # Stable hash ignores volatile Kafka wrapper offsets/source files; sorts payload keys before hashing.
    payload_canonical = F.to_json(F.map_from_entries(F.sort_array(F.map_entries(F.col("_business")))))
    p = (p.withColumn("_canonical_payload", payload_canonical)
        .withColumn("event_payload_hash", F.sha2(F.to_json(F.struct(
            "event_id_parsed", "event_type_parsed", "schema_version_parsed", "event_time_raw",
            "source_system", "run_id", "scenario", "_canonical_payload"),
            {"ignoreNullFields":"false"}), 256))
        .withColumn("quarantine_id", F.sha2(F.to_json(F.struct(
            "source_file", "kafka_offset", "raw_json_line"), {"ignoreNullFields":"false"}), 256)))

    reason = (F.when(F.col("_envelope").isNull() | F.col("_business").isNull(), "UNPARSEABLE_ENVELOPE_OR_PAYLOAD")
        .when(F.col("event_id_parsed").isNull() | (F.length("event_id_parsed")==0), "MISSING_EVENT_ID")
        .when(F.col("event_id").isNotNull() & (F.col("event_id") != F.col("event_id_parsed")), "RAW_EVENT_ID_MISMATCH")
        .when(~F.col("schema_version_parsed").isin("1", "1.0.0") |
              F.col("schema_version_parsed").isNull(), "UNSUPPORTED_SCHEMA_VERSION")
        .when(F.col("event_type_parsed").isNull() | (F.col("event_type_parsed") != "transaction.created"), "UNEXPECTED_EVENT_TYPE")
        .when(F.col("user_raw").isNotNull() & F.col("payload_user_numeric").isNull(), "INVALID_USER_ID")
        .when(F.col("user_id").isNull() | (F.col("user_id") < 0), "MISSING_CARD_IDENTITY")
        .when(F.col("kafka_user_numeric").isNotNull() & F.col("user_raw").isNotNull() &
              (F.col("payload_user_numeric") != F.col("kafka_user_numeric")), "KAFKA_USER_MISMATCH")
        .when(F.col("card_index").isNull() | (F.col("card_index") < 0), "MISSING_CARD_IDENTITY")
        .when(F.col("amount_raw").isNull() |
              (~F.trim("amount_raw").rlike(r"^(?:-?\$?|\$-)(?:[0-9]+|[0-9]{1,3}(?:,[0-9]{3})+)(?:\.[0-9]{1,2})?$")) |
              F.col("amount").isNull(), "BAD_AMOUNT")
        .when(F.col("currency").isNull() | (~F.col("currency").rlike(r"^[A-Z]{3}$")), "BAD_CURRENCY")
        .when(F.col("event_time_utc").isNull(), "BAD_EVENT_TIME")
        .when(F.col("source_event_time_local").isNull() & F.col("local_year").isNotNull(), "BAD_LOCAL_DATETIME")
        .when(F.col("mcc").isNull() | (F.col("mcc") < 0), "BAD_MCC")
        .when(F.col("synthetic_is_fraud").isNull(), "BAD_SYNTHETIC_TRUTH")
        .when((F.col("merchant_lat_raw").isNotNull() & F.col("merchant_latitude").isNull()) |
              (F.col("merchant_lon_raw").isNotNull() & F.col("merchant_longitude").isNull()) |
              (F.col("merchant_latitude").isNull() != F.col("merchant_longitude").isNull()) |
              (F.abs("merchant_latitude") > 90) | (F.abs("merchant_longitude") > 180) |
              F.isnan("merchant_latitude") | F.isnan("merchant_longitude"), "INVALID_COORDINATES")
        .otherwise(F.lit(None).cast("string")))
    return p.withColumn("quarantine_reason", reason)

## 3. Offline fixtures against Sayed's exact contract

These tests run the **same Spark transformation** as the production micro-batch. They check `Card: 0`, US-dollar formatting, signed refunds, truth separation, Kafka user mismatch, duplicate hashes independent of Kafka offsets, and bad coordinates. They do not write any tables.

Refund tests include the exact live S3 formatting `$-30.75` and `$-2,345.67`, which must be negative DECIMAL values, not fraudulent by sign alone.


In [0]:
import json
from datetime import datetime, timezone
now = datetime(2026, 10, 29, 14, 0, tzinfo=timezone.utc)
base = {
  "event_id":"fixture-001", "event_type":"transaction.created", "schema_version":1,
  "event_timestamp":"2026-10-29T14:35:10.120000+00:00", "source_system":"POS_TERMINAL",
  "run_id":"fixture-run", "scenario":"normal_spend",
  "payload": {"User":2042,"Card":0,"Year":2026,"Month":10,"Day":29,"Time":"14:35",
     "Amount":"$45.20","Use Chip":"Chip Transaction","Transaction Entry Method":"contactless_chip",
     "Merchant Name":12847192,"Merchant City":"Cleveland","Merchant State":"OH",
     "Merchant Latitude":41.4993,"Merchant Longitude":-81.6944,"Zip":"44101","MCC":5411,
     "Errors?":None,"Is Fraud?":"No"}
}
import copy
cases = []
def add(name, event, kafka_key="2042", as_encoded=False, kafka_offset=None):
    line = json.dumps({"key":kafka_key,"offset": 25 if kafka_offset is None else kafka_offset,
          "value":json.dumps(event) if as_encoded else event,"timestamp":"2026-10-29T14:35:10Z"})
    cases.append((name,line,f"s3://fixture/partition=0/{name}.json.gz",now,500,
                 event.get("event_id"),event.get("event_type"),str(event.get("schema_version")),
                 25 if kafka_offset is None else kafka_offset,kafka_key,now))
add("valid",base)
add("replay",base,kafka_offset=99)
a=copy.deepcopy(base);a["event_id"]="fixture-refund";a["payload"]["Amount"]="-$15.25";a["payload"]["Is Fraud?"]="Yes";add("refund",a,as_encoded=True)
a2=copy.deepcopy(base);a2["event_id"]="fixture-refund-dollar-before-minus";a2["payload"]["Amount"]="$-30.75";add("refund-dollar-minus",a2)
a3=copy.deepcopy(base);a3["event_id"]="fixture-refund-thousands";a3["payload"]["Amount"]="$-2,345.67";add("refund-thousands",a3)
a4=copy.deepcopy(base);a4["event_id"]="fixture-invalid-double-sign";a4["payload"]["Amount"]="$--30.75";add("invalid-double-sign",a4)
b=copy.deepcopy(base);b["event_id"]="fixture-bad";b["payload"]["Amount"]="not money";add("bad-amount",b)
c=copy.deepcopy(base);c["event_id"]="fixture-mismatch";add("bad-user",c,kafka_key="99")
d=copy.deepcopy(base);d["event_id"]="fixture-coordinates";d["payload"]["Merchant Longitude"]=None;add("bad-coordinate",d)
e=copy.deepcopy(base);e["event_id"]="fixture-legacy";e["schema_version"]="1.0.0";e["data"]={"user_id":2042,"card_id":0,"amount":"12.33","currency":"USD","mcc":5411,"is_fraud":False};del e["payload"];add("legacy",e)
fixture_schema=("case_name STRING, raw_json_line STRING, source_file STRING, source_modified_utc TIMESTAMP, "
  "source_size_bytes LONG, event_id STRING, event_type STRING, schema_version STRING, "
  "kafka_offset LONG, kafka_key STRING, ingested_at_utc TIMESTAMP")
fx_src=spark.createDataFrame(cases,fixture_schema)
fx_parsed=parse_sayed_raw(fx_src)
result={r.case_name:r for r in fx_parsed.select("case_name","quarantine_reason","user_id", "card_index", "amount",
        "currency","synthetic_is_fraud","event_payload_hash","event_time_utc").collect()}
assert result["valid"].quarantine_reason is None
assert result["valid"].user_id==2042 and result["valid"].card_index==0
assert str(result["valid"].amount)=="45.20" and result["valid"].currency=="USD"
assert result["refund"].quarantine_reason is None and str(result["refund"].amount)=="-15.25"
assert result["refund"].synthetic_is_fraud is True
assert result["refund-dollar-minus"].quarantine_reason is None
assert str(result["refund-dollar-minus"].amount)=="-30.75"
assert result["refund-dollar-minus"].currency=="USD"
assert result["refund-thousands"].quarantine_reason is None
assert str(result["refund-thousands"].amount)=="-2345.67"
assert result["invalid-double-sign"].quarantine_reason=="BAD_AMOUNT"
assert result["replay"].event_payload_hash==result["valid"].event_payload_hash
assert result["bad-amount"].quarantine_reason=="BAD_AMOUNT"
assert result["bad-user"].quarantine_reason=="KAFKA_USER_MISMATCH"
assert result["bad-coordinate"].quarantine_reason=="INVALID_COORDINATES"
assert result["legacy"].quarantine_reason is None
print("SAYED_CONTRACT_FIXTURES: PASS (new/legacy, Card=0, $- and -$ refunds, thousands, malformed signs, fraud, replay, coordinates)")

SAYED_CONTRACT_FIXTURES: PASS (new/legacy, Card=0, $- and -$ refunds, thousands, malformed signs, fraud, replay, coordinates)


## 4. Read-only real-data diagnostic (sample across latest AND earliest landing)

Only PASS if the real Raw dataset yields valid transactions using the verified parser, not just synthetic fixtures. This gate **does not** skip bad data: invalid records remain eligible for Quarantine after the stream runs.

The diagnostics also print `FINANCIAL_AMOUNT_AUDIT` to show dollar-before-minus and minus-before-dollar events and remaining rejected amounts.


In [0]:
# No writes. Examine both ends of the Raw history, because schema evolved.
latest = spark.table(RAW_TABLE).orderBy(F.col("ingested_at_utc").desc()).limit(250)
earliest = spark.table(RAW_TABLE).orderBy(F.col("ingested_at_utc").asc()).limit(250)
sample = latest.unionByName(earliest).dropDuplicates(["source_file", "kafka_offset", "raw_json_line"])
parsed_sample = parse_sayed_raw(sample)
display(parsed_sample.groupBy("schema_version_parsed","quarantine_reason").count().orderBy(F.desc("count")))
display(parsed_sample.groupBy("user_id_origin").agg(
  F.count("*").alias("rows"),
  F.sum(F.col("user_raw").isNotNull().cast("long")).alias("payload_user_present"),
  F.sum(F.col("card_raw").isNotNull().cast("long")).alias("card_present"),
  F.sum(F.col("amount").isNotNull().cast("long")).alias("amount_parsed"),
  F.sum(F.col("synthetic_is_fraud").isNotNull().cast("long")).alias("label_parsed")
))

financial_audit = parsed_sample.agg(
    F.sum(F.coalesce(F.col("amount_raw").rlike(r"^\s*\$-"), F.lit(False)).cast("long")).alias("dollar_before_minus"),
    F.sum(F.coalesce(F.col("amount_raw").rlike(r"^\s*-\$"), F.lit(False)).cast("long")).alias("minus_before_dollar"),
    F.sum(F.coalesce(F.col("amount") < 0, F.lit(False)).cast("long")).alias("parsed_negative_amounts"),
    F.sum(F.coalesce(F.col("quarantine_reason") == "BAD_AMOUNT", F.lit(False)).cast("long")).alias("rejected_bad_amounts")
).first().asDict()
print("FINANCIAL_AMOUNT_AUDIT:", financial_audit)
stats=parsed_sample.agg(F.count("*").alias("n"),
     F.sum(F.col("quarantine_reason").isNull().cast("long")).alias("valid_n")).first()
print("REAL_SOURCE_CONTRACT_SAMPLE:",stats.asDict())
if not stats.n or not stats.valid_n or stats.valid_n/stats.n < 0.50:
    raise RuntimeError("REAL_SOURCE_CONTRACT_FAILED: inspect diagnostics; Silver writes are blocked")
assert "synthetic_is_fraud" not in {"event_id","user_id","card_index","amount","currency"}
print("SILVER_PREFLIGHT: PASS — no tables written")

schema_version_parsed,quarantine_reason,count
1,null,500


user_id_origin,rows,payload_user_present,card_present,amount_parsed,label_parsed
PAYLOAD,500,500,500,500,500


FINANCIAL_AMOUNT_AUDIT: {'dollar_before_minus': 22, 'minus_before_dollar': 0, 'parsed_negative_amounts': 22, 'rejected_bad_amounts': 0}
REAL_SOURCE_CONTRACT_SAMPLE: {'n': 500, 'valid_n': 500}
SILVER_PREFLIGHT: PASS — no tables written


## 5. Publishing schema — SILVER only

A fresh **v2** output namespace; avoids the failed v1.x notebook output tables. The checkpoint is separately versioned. If there are published v2 rows but the v2 checkpoint is missing, this notebook FAILS rather than guessing the recovery path.

In [0]:
if RUN_STAGE == "SILVER":
    spark.sql(f"""CREATE TABLE IF NOT EXISTS {TX_TABLE} (
      event_id STRING, event_type STRING, schema_version STRING,
      transaction_id STRING, source_transaction_id STRING, transaction_id_origin STRING,
      user_id BIGINT, user_id_origin STRING, card_index BIGINT, scoped_card_key STRING,
      amount DECIMAL(18,2), currency STRING, currency_origin STRING,
      merchant_name STRING, merchant_city STRING, merchant_state STRING,
      merchant_latitude DOUBLE, merchant_longitude DOUBLE, zip STRING, mcc INT,
      use_chip STRING, transaction_entry_method STRING, errors STRING,
      source_system STRING, event_time_utc TIMESTAMP, source_event_time_local TIMESTAMP_NTZ,
      event_payload_hash STRING, source_file STRING, source_modified_utc TIMESTAMP,
      kafka_partition INT, kafka_offset BIGINT, kafka_key STRING, ingested_at_utc TIMESTAMP,
      ingestion_lag_seconds DOUBLE, processing_lag_seconds DOUBLE
    ) USING DELTA""")
    
    spark.sql(f"""CREATE TABLE IF NOT EXISTS {ENRICHED_TX_TABLE} (
      event_id STRING, event_type STRING, schema_version STRING,
      transaction_id STRING, source_transaction_id STRING, transaction_id_origin STRING,
      user_id BIGINT, user_id_origin STRING, card_index BIGINT, scoped_card_key STRING,
      amount DECIMAL(18,2), currency STRING, currency_origin STRING,
      merchant_name STRING, merchant_city STRING, merchant_state STRING,
      merchant_latitude DOUBLE, merchant_longitude DOUBLE, zip STRING, mcc INT,
      use_chip STRING, transaction_entry_method STRING, errors STRING,
      source_system STRING, event_time_utc TIMESTAMP, source_event_time_local TIMESTAMP_NTZ,
      event_payload_hash STRING, source_file STRING, source_modified_utc TIMESTAMP,
      kafka_partition INT, kafka_offset BIGINT, kafka_key STRING, ingested_at_utc TIMESTAMP,
      ingestion_lag_seconds DOUBLE, processing_lag_seconds DOUBLE,
      customer_id STRING, user_city STRING, user_region STRING,
      user_fico_score INT, user_age INT, user_num_cards INT, user_total_debt DECIMAL(18,2),
      card_brand STRING, card_type STRING, credit_limit DECIMAL(18,2),
      is_enriched BOOLEAN, enrichment_status STRING
    ) USING DELTA""")

    spark.sql(f"""CREATE TABLE IF NOT EXISTS {TRUTH_TABLE} (
      event_id STRING, synthetic_is_fraud BOOLEAN, event_payload_hash STRING,
      scenario STRING, run_id STRING
    ) USING DELTA""")
    
    spark.sql(f"""CREATE TABLE IF NOT EXISTS {QUAR_TABLE} (
      quarantine_id STRING, event_id STRING, quarantine_reason STRING,
      raw_json_line STRING, source_file STRING, kafka_offset BIGINT,
      ingested_at_utc TIMESTAMP, remediation_status STRING
    ) USING DELTA""")
    
    spark.sql(f"""CREATE TABLE IF NOT EXISTS {REG_TABLE} (
      event_id STRING, event_payload_hash STRING, source_file STRING,
      first_ingested_at_utc TIMESTAMP
    ) USING DELTA""")

    spark.sql(f"""CREATE TABLE IF NOT EXISTS {PREDICTIONS_TABLE} (
      event_id STRING, prediction_score DOUBLE, predicted_label BOOLEAN,
      latency_ms DOUBLE, model_version STRING, predicted_at_utc TIMESTAMP
    ) USING DELTA""")

    spark.sql(f"""CREATE TABLE IF NOT EXISTS {DRIFT_METRICS_TABLE} (
      reconciled_at_utc TIMESTAMP, batch_id BIGINT, total_evaluated BIGINT,
      true_positives BIGINT, false_positives BIGINT, true_negatives BIGINT, false_negatives BIGINT,
      precision DOUBLE, recall DOUBLE, f1_score DOUBLE, mean_latency_ms DOUBLE
    ) USING DELTA""")

    expected = {TX_TABLE: {"event_id", "user_id", "card_index", "amount", "event_payload_hash"},
                ENRICHED_TX_TABLE: {"event_id", "user_id", "user_fico_score", "card_brand", "is_enriched"},
                TRUTH_TABLE: {"event_id", "synthetic_is_fraud", "event_payload_hash"},
                QUAR_TABLE: {"quarantine_id", "quarantine_reason"},
                REG_TABLE: {"event_id", "event_payload_hash"}}
    for name, columns in expected.items():
        found = set(spark.table(name).columns)
        if columns - found:
            raise RuntimeError(f"SILVER_TABLE_SCHEMA_MISMATCH {name}: {sorted(columns-found)}")
    try:
        dbutils.fs.ls(SILVER_CHECKPOINT)
        has_checkpoint = True
    except Exception:
        has_checkpoint = False
    print("SILVER_V3_OUTPUTS_READY", "checkpoint_present:", has_checkpoint)
else:
    print("PREFLIGHT: No Silver output tables created")


SILVER_V2_OUTPUTS_READY checkpoint_present False


## 6. Spark Connect-safe microbatch MERGE — replay, dedup, conflict detection

Batch source → parse → stop on contract drift → stop on conflicting same-event content → idempotent MERGE into Transactions, Truth, Quarantine and Registry. Each MERGE is atomic individually; the **batch checkpoint only commits after ALL** successful writes. Never run more than one Silver query on this checkpoint.

In [0]:
def silver_v2_microbatch(batch_df, batch_id):
    session = batch_df.sparkSession
    TX = TX_TABLE
    TRUTH = TRUTH_TABLE
    QUAR = QUAR_TABLE
    REG = REG_TABLE
    ENRICHED_TX = ENRICHED_TX_TABLE
    PREDS = PREDICTIONS_TABLE
    DRIFT = DRIFT_METRICS_TABLE
    DIM_U = DIM_USERS_TABLE
    DIM_C = DIM_CARDS_TABLE

    def parse_sayed_raw(frame):
        # Local definition ensures pure closure across executors
        def get(k):
            return F.coalesce(
                F.get_json_object(F.col("_business"), f"$.{k}"),
                F.get_json_object(F.col("_business"), f"$.['{k}']"),
                F.get_json_object(F.col("_business"), f"$.\"{k}\"")
            )

        p = (frame
            .withColumn("_event_json_str", F.get_json_object(F.col("raw_json_line"), "$.value"))
            .withColumn("_is_kafka_wrapper", F.col("_event_json_str").isNotNull())
            .withColumn("_envelope", F.when(F.col("_is_kafka_wrapper"), F.col("_event_json_str"))
                                        .otherwise(F.col("raw_json_line")))
            .withColumn("event_id_parsed", F.coalesce(F.get_json_object(F.col("_envelope"), "$.event_id"), F.col("event_id")))
            .withColumn("event_type_parsed", F.coalesce(F.get_json_object(F.col("_envelope"), "$.event_type"), F.col("event_type")))
            .withColumn("schema_version_parsed", F.coalesce(F.get_json_object(F.col("_envelope"), "$.schema_version"), F.col("schema_version")))
            .withColumn("event_time_raw", F.coalesce(
                F.get_json_object(F.col("_envelope"), "$.event_timestamp"),
                F.get_json_object(F.col("_envelope"), "$.event_time"),
                F.get_json_object(F.col("_envelope"), "$.timestamp")))
            .withColumn("source_system", F.get_json_object(F.col("_envelope"), "$.source_system"))
            .withColumn("run_id", F.get_json_object(F.col("_envelope"), "$.run_id"))
            .withColumn("scenario", F.get_json_object(F.col("_envelope"), "$.scenario"))
            .withColumn("_payload_candidate", F.get_json_object(F.col("_envelope"), "$.payload"))
            .withColumn("_data_candidate", F.get_json_object(F.col("_envelope"), "$.data"))
            .withColumn("_business", F.coalesce(F.col("_payload_candidate"), F.col("_data_candidate"))))

        p = (p.withColumn("user_raw", F.coalesce(get("User"), get("user_id"), get("user")))
            .withColumn("card_raw", F.coalesce(get("Card"), get("card_id"), get("card_index"), get("card_idx")))
            .withColumn("amount_raw", F.coalesce(get("Amount"), get("amount"), get("transaction_amount")))
            .withColumn("fraud_raw", F.coalesce(get("Is Fraud?"), get("is_fraud"), get("fraud_label")))
            .withColumn("mcc_raw", F.coalesce(get("MCC"), get("mcc"), get("merchant_category_code")))
            .withColumn("merchant_name", F.coalesce(get("Merchant Name"), get("merchant_name")))
            .withColumn("merchant_city", F.coalesce(get("Merchant City"), get("merchant_city")))
            .withColumn("merchant_state", F.coalesce(get("Merchant State"), get("merchant_state")))
            .withColumn("merchant_lat_raw", F.coalesce(get("Merchant Latitude"), get("merchant_latitude")))
            .withColumn("merchant_lon_raw", F.coalesce(get("Merchant Longitude"), get("merchant_longitude")))
            .withColumn("zip_raw", F.coalesce(get("Zip"), get("zip"), get("zip_code")))
            .withColumn("use_chip", F.coalesce(get("Use Chip"), get("use_chip")))
            .withColumn("transaction_entry_method", F.coalesce(get("Transaction Entry Method"), get("transaction_entry_method")))
            .withColumn("errors", F.coalesce(get("Errors?"), get("errors")))
            .withColumn("currency_raw", F.coalesce(get("currency"), get("Currency")))
            .withColumn("source_transaction_id", F.coalesce(get("transaction_id"), get("tx_id")))
            .withColumn("local_year", get("Year"))
            .withColumn("local_month", get("Month"))
            .withColumn("local_day", get("Day"))
            .withColumn("local_clock", get("Time")))

        p = (p.withColumn("payload_user_numeric", F.expr("try_cast(trim(user_raw) AS BIGINT)"))
            .withColumn("kafka_user_numeric", F.expr("try_cast(trim(kafka_key) AS BIGINT)"))
            .withColumn("user_id", F.when(F.col("user_raw").isNotNull(), F.col("payload_user_numeric"))
                    .otherwise(F.col("kafka_user_numeric")))
            .withColumn("user_id_origin", F.when(F.col("user_raw").isNotNull(), F.lit("PAYLOAD"))
                    .when(F.col("kafka_user_numeric").isNotNull(), F.lit("KAFKA_KEY_VERIFIED_FALLBACK"))
                    .otherwise(F.lit("UNKNOWN")))
            .withColumn("card_index", F.expr("try_cast(trim(card_raw) AS BIGINT)"))
            .withColumn("mcc", F.expr("try_cast(trim(mcc_raw) AS INT)"))
            .withColumn("merchant_latitude", F.expr("try_cast(trim(merchant_lat_raw) AS DOUBLE)"))
            .withColumn("merchant_longitude", F.expr("try_cast(trim(merchant_lon_raw) AS DOUBLE)"))
            .withColumn("zip", F.regexp_replace(F.trim("zip_raw"), r"\.0$", ""))
            .withColumn("event_time_utc", F.expr("try_to_timestamp(event_time_raw)")))

        p = (p.withColumn("currency", F.when(F.col("currency_raw").isNotNull(), F.upper(F.trim("currency_raw")))
                      .when(F.col("amount_raw").rlike(r"^\s*(?:-?\$|\$-)"), F.lit("USD"))
                      .otherwise(F.lit(None).cast("string")))
            .withColumn("currency_origin", F.when(F.col("currency_raw").isNotNull(), "EXPLICIT")
                      .when(F.col("amount_raw").rlike(r"^\s*(?:-?\$|\$-)"), "IMPLIED_DOLLAR_SIGN")
                      .otherwise("UNKNOWN"))
            .withColumn("amount_stripped", F.regexp_replace(F.trim("amount_raw"), r"[\$,]", ""))
            .withColumn("amount", F.expr("try_cast(amount_stripped AS DECIMAL(18,2))"))
            .withColumn("fraud_norm", F.lower(F.trim("fraud_raw")))
            .withColumn("synthetic_is_fraud", F.when(F.col("fraud_norm").isin("yes", "true", "1"), F.lit(True))
                      .when(F.col("fraud_norm").isin("no", "false", "0"), F.lit(False))
                      .otherwise(F.lit(None).cast("boolean")))
            .withColumn("scoped_card_key", F.concat_ws(":", "user_id", "card_index"))
            .withColumn("kafka_partition", F.expr("try_cast(regexp_extract(source_file, 'partition=([0-9]+)', 1) AS INT)"))
            .withColumn("transaction_id_origin", F.when(F.length(F.trim("source_transaction_id"))>0, "SOURCE")
                      .otherwise("DERIVED_FROM_EVENT_ID"))
            .withColumn("transaction_id", F.when(F.col("transaction_id_origin")=="SOURCE", F.col("source_transaction_id"))
                      .otherwise(F.concat(F.lit("EVENT_"), F.col("event_id_parsed")))))

        local_text = F.concat(F.lpad(F.trim("local_year"), 4, "0"), F.lit("-"),
                              F.lpad(F.trim("local_month"), 2, "0"), F.lit("-"),
                              F.lpad(F.trim("local_day"), 2, "0"), F.lit(" "),
                              F.when(F.trim("local_clock").rlike(r"^[0-9]{2}:[0-9]{2}$"),
                                     F.concat(F.trim("local_clock"), F.lit(":00")))
                               .otherwise(F.trim("local_clock")))
        p = p.withColumn("_local_text", local_text).withColumn(
            "source_event_time_local", F.expr("try_cast(_local_text AS TIMESTAMP_NTZ)"))

        payload_canonical = F.to_json(F.map_from_entries(F.sort_array(F.map_entries(F.col("_business")))))
        p = (p.withColumn("_canonical_payload", payload_canonical)
            .withColumn("event_payload_hash", F.sha2(F.to_json(F.struct(
                "event_id_parsed", "event_type_parsed", "schema_version_parsed", "event_time_raw",
                "source_system", "run_id", "scenario", "_canonical_payload"),
                {"ignoreNullFields": "false"}), 256))
            .withColumn("quarantine_id", F.sha2(F.to_json(F.struct(
                "source_file", "kafka_offset", "raw_json_line"), {"ignoreNullFields": "false"}), 256)))

        reason = (F.when(F.col("_envelope").isNull() | F.col("_business").isNull(), "UNPARSEABLE_ENVELOPE_OR_PAYLOAD")
            .when(F.col("event_id_parsed").isNull() | (F.length("event_id_parsed")==0), "MISSING_EVENT_ID")
            .when(F.col("event_id").isNotNull() & (F.col("event_id") != F.col("event_id_parsed")), "RAW_EVENT_ID_MISMATCH")
            .when(~F.col("schema_version_parsed").isin("1", "1.0.0") |
                  F.col("schema_version_parsed").isNull(), "UNSUPPORTED_SCHEMA_VERSION")
            .when(F.col("event_type_parsed").isNull() | (F.col("event_type_parsed") != "transaction.created"), "UNEXPECTED_EVENT_TYPE")
            .when(F.col("user_raw").isNotNull() & F.col("payload_user_numeric").isNull(), "INVALID_USER_ID")
            .when(F.col("user_id").isNull() | (F.col("user_id") < 0), "MISSING_CARD_IDENTITY")
            .when(F.col("kafka_user_numeric").isNotNull() & F.col("user_raw").isNotNull() &
                  (F.col("payload_user_numeric") != F.col("kafka_user_numeric")), "KAFKA_USER_MISMATCH")
            .when(F.col("card_index").isNull() | (F.col("card_index") < 0), "MISSING_CARD_IDENTITY")
            .when(F.col("amount_raw").isNull() |
                  (~F.trim("amount_raw").rlike(r"^(?:-?\$?|\$-)(?:[0-9]+|[0-9]{1,3}(?:,[0-9]{3})+)(?:\.[0-9]{1,2})?$")) |
                  F.col("amount").isNull(), "BAD_AMOUNT")
            .when(F.col("currency").isNull() | (~F.col("currency").rlike(r"^[A-Z]{3}$")), "BAD_CURRENCY")
            .when(F.col("event_time_utc").isNull(), "BAD_EVENT_TIME")
            .when(F.col("source_event_time_local").isNull() & F.col("local_year").isNotNull(), "BAD_LOCAL_DATETIME")
            .when(F.col("mcc").isNull() | (F.col("mcc") < 0), "BAD_MCC")
            .when(F.col("synthetic_is_fraud").isNull(), "BAD_SYNTHETIC_TRUTH")
            .when((F.col("merchant_lat_raw").isNotNull() & F.col("merchant_latitude").isNull()) |
                  (F.col("merchant_lon_raw").isNotNull() & F.col("merchant_longitude").isNull()) |
                  (F.col("merchant_latitude").isNull() != F.col("merchant_longitude").isNull()) |
                  (F.abs("merchant_latitude") > 90) | (F.abs("merchant_longitude") > 180) |
                  F.isnan("merchant_latitude") | F.isnan("merchant_longitude"), "INVALID_COORDINATES")
            .otherwise(F.lit(None).cast("string")))
        return p.withColumn("quarantine_reason", reason)

    parsed = parse_sayed_raw(batch_df).persist()
    try:
        totals = parsed.agg(
            F.count("*").alias("n"),
            F.sum(F.col("quarantine_reason").isNull().cast("long")).alias("valid_n")
        ).first()
        n = int(totals.n)
        if n == 0:
            print(f"SILVER_EMPTY_BATCH batch={batch_id}")
            return
        good = int(totals.valid_n or 0)
        if good == 0 or good / n < 0.50:
            raise RuntimeError(f"SILVER_CONTRACT_DRIFT batch={batch_id} valid={good} total={n}; stop before write")

        # Reject conflicting event IDs within batch
        keyed = parsed.filter(F.col("event_id_parsed").isNotNull() & (F.length("event_id_parsed") > 0))
        conflict = keyed.groupBy("event_id_parsed").agg(
            F.countDistinct("event_payload_hash").alias("hashes")).filter("hashes > 1").limit(1).count()
        if conflict:
            raise RuntimeError(f"EVENT_ID_CONFLICT_IN_BATCH batch={batch_id}")
        
        reg_w = Window.partitionBy("event_id_parsed").orderBy(
            F.col("source_file").asc_nulls_last(), F.col("kafka_offset").asc_nulls_last())
        reg_src = (keyed.withColumn("rn_reg", F.row_number().over(reg_w)).filter("rn_reg=1")
            .select(F.col("event_id_parsed").alias("event_id"), "event_payload_hash",
                    "source_file", F.col("ingested_at_utc").alias("first_ingested_at_utc")))
        
        conflict_stored = (reg_src.alias("i").join(session.table(REG).alias("r"), "event_id", "inner")
             .filter(F.col("i.event_payload_hash") != F.col("r.event_payload_hash")).limit(1).count())
        if conflict_stored:
            raise RuntimeError(f"EVENT_ID_CONFLICT_AGAINST_REGISTRY batch={batch_id}")

        w = Window.partitionBy("event_id_parsed").orderBy(
             F.col("source_file").asc_nulls_last(), F.col("kafka_offset").asc_nulls_last())
        good_src = (parsed.filter(F.col("quarantine_reason").isNull())
                    .withColumn("rn", F.row_number().over(w)).filter("rn=1").drop("rn"))
        
        # Calculate in-stream Lag Telemetry
        good_src = (good_src
            .withColumn("ingestion_lag_seconds",
                F.round(F.unix_timestamp(F.current_timestamp()) - F.unix_timestamp(F.col("event_time_utc")), 2))
            .withColumn("processing_lag_seconds",
                F.round(F.unix_timestamp(F.current_timestamp()) - F.unix_timestamp(F.col("source_modified_utc")), 2)))

        tx = good_src.select(
            F.col("event_id_parsed").alias("event_id"),
            F.col("event_type_parsed").alias("event_type"),
            F.col("schema_version_parsed").alias("schema_version"),
            "transaction_id", "source_transaction_id", "transaction_id_origin",
            "user_id", "user_id_origin", "card_index", "scoped_card_key", "amount", "currency", "currency_origin",
            "merchant_name", "merchant_city", "merchant_state", "merchant_latitude", "merchant_longitude",
            "zip", "mcc", "use_chip", "transaction_entry_method", "errors", "source_system",
            "event_time_utc", "source_event_time_local", "event_payload_hash", "source_file",
            "source_modified_utc", "kafka_partition", "kafka_offset", "kafka_key", "ingested_at_utc",
            "ingestion_lag_seconds", "processing_lag_seconds"
        )
        
        truth = good_src.select(
            F.col("event_id_parsed").alias("event_id"),
            "synthetic_is_fraud", "event_payload_hash", "scenario", "run_id"
        )
        
        bad = (parsed.filter(F.col("quarantine_reason").isNotNull())
               .withColumn("remediation_status", F.lit("NEW"))
               .select("quarantine_id", F.col("event_id_parsed").alias("event_id"),
                       "quarantine_reason", "raw_json_line", "source_file", "kafka_offset",
                       "ingested_at_utc", "remediation_status")
               .dropDuplicates(["quarantine_id"]))

        # CDC Dimension Enrichment (Broadcast Join)
        try:
            dim_u = session.table(DIM_U).select(
                F.col("source_user_index").alias("dim_user_id"),
                F.col("customer_id").alias("dim_customer_id"),
                F.col("customer_city").alias("user_city"),
                F.col("customer_region").alias("user_region"),
                F.col("fico_score").alias("user_fico_score"),
                F.col("current_age").alias("user_age"),
                F.col("num_credit_cards").alias("user_num_cards"),
                F.col("total_debt").alias("user_total_debt")
            )
            dim_c = session.table(DIM_C).select(
                F.col("customer_id").alias("dim_c_cust_id"),
                F.col("source_card_index").cast("bigint").alias("dim_c_card_idx"),
                F.col("card_brand"),
                F.col("card_type"),
                F.col("credit_limit")
            )
            tx_enriched = (tx.join(F.broadcast(dim_u), tx["user_id"] == dim_u["dim_user_id"], "left")
                             .join(F.broadcast(dim_c),
                                   (F.col("dim_customer_id") == F.col("dim_c_cust_id")) &
                                   (tx["card_index"] == F.col("dim_c_card_idx")), "left")
                             .withColumn("customer_id", F.col("dim_customer_id"))
                             .withColumn("is_enriched", F.col("user_fico_score").isNotNull())
                             .withColumn("enrichment_status",
                                         F.when(F.col("user_fico_score").isNotNull(), F.lit("ENRICHED"))
                                          .otherwise(F.lit("PENDING_USER_CDC")))
                             .drop("dim_user_id", "dim_customer_id", "dim_c_cust_id", "dim_c_card_idx"))
        except Exception as enrich_exc:
            print("CDC_ENRICHMENT_DIAGNOSTIC:", str(enrich_exc)[:200])
            tx_enriched = (tx.withColumn("customer_id", F.lit(None).cast("string"))
                             .withColumn("user_city", F.lit(None).cast("string"))
                             .withColumn("user_region", F.lit(None).cast("string"))
                             .withColumn("user_fico_score", F.lit(None).cast("int"))
                             .withColumn("user_age", F.lit(None).cast("int"))
                             .withColumn("user_num_cards", F.lit(None).cast("int"))
                             .withColumn("user_total_debt", F.lit(None).cast("decimal(18,2)"))
                             .withColumn("card_brand", F.lit(None).cast("string"))
                             .withColumn("card_type", F.lit(None).cast("string"))
                             .withColumn("credit_limit", F.lit(None).cast("decimal(18,2)"))
                             .withColumn("is_enriched", F.lit(False))
                             .withColumn("enrichment_status", F.lit("LOOKUP_OFFLINE")))

        def insert_only(target, source, view, key):
            source.createOrReplaceTempView(view)
            cols = source.columns
            columns = ", ".join(f"`{col}`" for col in cols)
            values = ", ".join(f"s.`{col}`" for col in cols)
            session.sql(f"MERGE INTO {target} AS t USING {view} AS s ON t.`{key}`=s.`{key}` "
                        f"WHEN NOT MATCHED THEN INSERT ({columns}) VALUES ({values})").collect()

        insert_only(TX, tx, "_sic8_v2_tx_batch", "event_id")
        insert_only(ENRICHED_TX, tx_enriched, "_sic8_v2_tx_enriched_batch", "event_id")
        insert_only(TRUTH, truth, "_sic8_v2_truth_batch", "event_id")
        insert_only(QUAR, bad, "_sic8_v2_quar_batch", "quarantine_id")
        insert_only(REG, reg_src, "_sic8_v2_reg_batch", "event_id")

        # Optional Model Predictions Reconciliation (if predictions are present)
        try:
            if session.catalog.tableExists(PREDS) and session.catalog.tableExists(DRIFT):
                preds_df = session.table(PREDS)
                rec = preds_df.join(truth, "event_id", "inner").select(
                    "event_id", F.col("predicted_label"), F.col("synthetic_is_fraud").alias("actual_label"), "latency_ms"
                )
                rec_cnt = rec.count()
                if rec_cnt > 0:
                    tp = rec.filter("predicted_label = true and actual_label = true").count()
                    fp = rec.filter("predicted_label = true and actual_label = false").count()
                    tn = rec.filter("predicted_label = false and actual_label = false").count()
                    fn = rec.filter("predicted_label = false and actual_label = true").count()
                    prec = (tp / (tp + fp)) if (tp + fp) > 0 else 0.0
                    reca = (tp / (tp + fn)) if (tp + fn) > 0 else 0.0
                    f1 = (2 * prec * reca / (prec + reca)) if (prec + reca) > 0 else 0.0
                    mean_lat = rec.agg(F.avg("latency_ms")).first()[0] or 0.0
                    drift_row = session.createDataFrame([{
                        "reconciled_at_utc": F.current_timestamp(),
                        "batch_id": int(batch_id),
                        "total_evaluated": rec_cnt,
                        "true_positives": tp,
                        "false_positives": fp,
                        "true_negatives": tn,
                        "false_negatives": fn,
                        "precision": float(prec),
                        "recall": float(reca),
                        "f1_score": float(f1),
                        "mean_latency_ms": float(mean_lat)
                    }])
                    drift_row.write.format("delta").mode("append").saveAsTable(DRIFT)
        except Exception as pred_exc:
            pass

        avg_lag = good_src.agg(F.avg("ingestion_lag_seconds")).first()[0] if good > 0 else 0.0
        print(f"SILVER_BATCH_SUCCESS batch={batch_id} raw={n} accepted={good} invalid={n-good} avg_lag_sec={avg_lag}")
    finally:
        parsed.unpersist()


## 7. Execute Silver stream (only after preflight PASS)

`availableNow=True` consumes the current Raw Delta backlog, commits its own checkpoint, then exits. Schedule it **after** 08A in Databricks Lakeflow Jobs, preserving both checkpoints. Never run simultaneously with another instance of 08B.

In [0]:
if RUN_STAGE == "SILVER":
    raw_stream = spark.readStream.format("delta").table(RAW_TABLE)
    trigger_arg = {"processingTime": TRIGGER_INTERVAL} if STREAMING_MODE == "CONTINUOUS" else {"availableNow": True}
    query = (raw_stream.writeStream.foreachBatch(silver_v2_microbatch)
             .option("checkpointLocation", SILVER_CHECKPOINT)
             .trigger(**trigger_arg).start())
    print(f"SILVER_STREAM_STARTED (mode={STREAMING_MODE}, trigger={trigger_arg})")
    if STREAMING_MODE == "AVAILABLE_NOW":
        query.awaitTermination()
        if query.exception() is not None:
            raise RuntimeError("SILVER_STREAM_ERROR: " + str(query.exception()))
        print("SILVER_STREAM_COMPLETED", query.lastProgress)
    else:
        print("Silver continuous stream is active. Awaiting new micro-batches...")
        query.awaitTermination()
else:
    print("PREFLIGHT: no streaming query started")


SILVER_STREAM_COMPLETED {
    "id": "60bcd23c-0c13-4030-985a-7a55160ae462",
    "runId": "74d142e1-36be-44a4-ae39-ee344ddcf26f",
    "name": null,
    "timestamp": "2026-10-29T16:12:33.529Z",
    "batchId": 0,
    "batchDuration": 42175,
    "durationMs": {
        "triggerExecution": 42168,
        "queryPlanning": 235,
        "collectSourceMetrics": 79,
        "getBatch": 61,
        "commitOffsets": 206,
        "addBatch": 40138,
        "latestOffset": 1201,
        "walCommit": 171
    },
    "eventTime": {},
    "stateOperators": [],
    "sources": [
        {
            "description": "DeltaSource[s3://dbstorage-prod-oubgdtr06h/uc/27fb8cb0-d62b-4c3a-bb55-4ca67c05ff29/95db93ae-b5ca-4b03-8e54-522867adb8f8/__unitystorage/catalogs/c31eb016-f4f2-40d2-b080-e203596b1a90/tables/0584a06d-b0e1-4bc7-ad45-4a3542e295e0]",
            "startOffset": null,
            "endOffset": "{\"sourceVersion\":1,\"reservoirId\":\"761c5037-dd98-4c6e-847b-252087cb26f7\",\"reservoirVersion\":14,\"index

## 8. Verify Published Tables (read-only)

Requires exactly one non-leaking Truth record per accepted event, no duplicate keys, and no disagreement between registry and transactions. Quarantine is independently keyed and can include events that were never accepted. Any failure must be investigated, not suppressed.

In [0]:
if RUN_STAGE == "SILVER":
    counts = {t:spark.table(t).count() for t in [TX_TABLE,TRUTH_TABLE,QUAR_TABLE,REG_TABLE]}
    print("SILVER_V2_COUNTS",counts)
    if counts[TX_TABLE]!=counts[TRUTH_TABLE]:
        raise RuntimeError("DQ_FAIL: not every accepted event has exactly one ground-truth row")
    for t,col in [(TX_TABLE,"event_id"),(TRUTH_TABLE,"event_id"),(REG_TABLE,"event_id"),(QUAR_TABLE,"quarantine_id")]:
        if spark.table(t).groupBy(col).count().filter("count>1").limit(1).count():
            raise RuntimeError(f"DQ_FAIL: duplicate {col} in {t}")
    missing=spark.table(TX_TABLE).select("event_id").join(
           spark.table(TRUTH_TABLE).select("event_id"),"event_id","left_anti").limit(1).count()
    if missing:raise RuntimeError("DQ_FAIL: accepted transaction without truth")
    bad_hash=(spark.table(TX_TABLE).alias("t").join(spark.table(TRUTH_TABLE).alias("y"),"event_id")
        .filter(F.col("t.event_payload_hash")!=F.col("y.event_payload_hash")).limit(1).count())
    if bad_hash:raise RuntimeError("DQ_FAIL: transaction and ground truth payload hashes disagree")
    forbidden={"synthetic_is_fraud","fraud_raw","is_fraud","raw_json_line","scenario","run_id"}
    if forbidden & set(spark.table(TX_TABLE).columns):
        raise RuntimeError("LABEL_LEAKAGE_FAIL: training surface has labels, raw JSON or simulator scenario")
    print("SILVER_V2_VERIFICATION: PASS")
    display(spark.table(QUAR_TABLE).groupBy("quarantine_reason").count().orderBy(F.desc("count")))
else:
    print("PREFLIGHT complete. If tests PASS, set RUN_STAGE='SILVER' and Run All.")

SILVER_V2_COUNTS {'workspace.sic8_streaming.streaming_transactions_v2': 275151, 'workspace.sic8_streaming.streaming_synthetic_truth_v2': 275151, 'workspace.sic8_streaming.streaming_quarantine_v2': 0, 'workspace.sic8_streaming.streaming_event_registry_v2': 275151}
SILVER_V2_VERIFICATION: PASS


quarantine_reason,count


## 9. Operations handoff (no extra notebook changes)

- **R1/Sayed:** Kafka → Aiven S3 Sink `s3://sic-pipeline-storage/bronze/transactions/`, separate Kafka → Redis/LightGBM consumer. Never send synthetic `Is Fraud?` to the fraud model.
- **R2:** 08A Raw INGEST → this 08B Silver `availableNow` runs, in order. Schedule a single Databricks Lakeflow Job with `08A → 08B` dependency and `max_concurrent_runs=1`, using the same stable checkpoints. Source freshness depends on S3 sink flushing files.
- **AWS orchestration:** keep ECS/EventBridge and existing scheduled reference exports under AWS. Do **not** create another competing scheduler for the same work.
- **Not yet covered:** User/Card CDC, prediction/alert archive, streaming Gold KPIs, end-to-end automated freshness/lag, model evaluation. These require separate validated pipelines/contracts.
- **Caution:** In a failure during multiple MERGEs, Delta tables can temporarily disagree until the same uncommitted microbatch is retried. Do not delete checkpoint or force new run with changed output names.